# Tugas Besar ITS — Langkah 1–4
**Tema: Sistem Rekomendasi Adaptif Materi & Jalur Belajar**  
Kelas / kelompok: **isi kelompok**  
Anggota dan NPM: **isi seluruh anggota**

Notebook mempertahankan eksplorasi awal dan melanjutkannya dengan preprocessing, TF-IDF + cosine similarity, serta integrasi XP, badge dan quest/progres. Fuzzy Matching dan Rabin-Karp tidak digunakan karena bukan algoritma utama tema kelompok ini. Tidak ada frontend atau REST API (langkah 5–6).

**Cara menjalankan:** unggah notebook ke Google Colab, jalankan dari atas, lalu unggah satu CSV MOOC ketika diminta. GPU dan Google Drive tidak diperlukan. Pada sel terakhir, unduh ZIP berisi hasil, model dan dokumentasi. Mengulang seluruh notebook akan mengulang simulasi dari awal.

**Batas data:** MOOC adalah katalog kursus, bukan nilai/riwayat siswa. Nilai kuis di bagian gamifikasi merupakan **simulasi**, bukan data MOOC. Aturan adaptasi adalah aturan prototipe kelompok yang perlu validasi pengajar; daftar rekomendasi bukan prasyarat pedagogis resmi.

## 1. Tujuan dan hubungan dengan tema
Tujuan eksplorasi adalah memastikan apakah nama kursus, deskripsi, dan informasi keterampilan dapat menjadi korpus rekomendasi berbasis isi. Pada langkah berikutnya, teks dapat direpresentasikan dengan TF-IDF dan dibandingkan dengan kebutuhan belajar menggunakan cosine similarity, sesuai algoritma tema pada panduan.

`Difficulty Level` dapat dipertimbangkan sebagai metadata penyaring setelah makna kategorinya divalidasi. Rating merupakan penilaian kursus, **bukan nilai kemampuan siswa**. Urutan Beginner → Intermediate → Advanced belum membuktikan hubungan prasyarat antar kursus.

**Provenans:** sumber langsung adalah file yang diunggah kelompok. URL dalam data mengarah ke katalog Coursera, tetapi pembuat CSV, URL sumber unduhan, tanggal pengumpulan, dan lisensinya belum terverifikasi. Lengkapi informasi itu dari tempat kelompok memperoleh dataset; jangan menganggap tanggal menjalankan notebook sebagai tanggal pengumpulan data.

In [ ]:
from pathlib import Path
from io import BytesIO
from collections import Counter
from urllib.parse import urlparse
import hashlib, json, re, sys, platform, shutil
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

pd.set_option('display.max_colwidth', 100)
pd.set_option('display.max_columns', 15)
plt.rcParams.update({'figure.figsize': (10, 5), 'axes.spines.top': False,
                     'axes.spines.right': False, 'font.size': 11})
OUTPUT_DIR = Path('hasil_langkah_1')
OUTPUT_DIR.mkdir(exist_ok=True)
print('Python:', platform.python_version(), '| pandas:', pd.__version__)
print('Folder hasil:', OUTPUT_DIR.resolve())

## 2. Membaca CSV dan memeriksa encoding
CSV yang diberikan tidak dapat dibaca sebagai UTF-8 biasa. Kode mencoba UTF-8 terlebih dahulu lalu Windows-1252 (`cp1252`) secara ketat, tanpa membuang byte yang tidak dikenal. Pemilihan encoding ini untuk membaca file, bukan normalisasi teks.

Di luar Colab, letakkan CSV satu folder dengan notebook atau isi `LOCAL_CSV`. Nilai asli dipertahankan, termasuk string kosong; audit di bawah membedakan sel kosong dan penanda missing berbentuk teks.

In [ ]:
LOCAL_CSV = ''  # Opsional: path CSV untuk menjalankan notebook di luar Colab.
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    uploaded = files.upload()
    candidates = [name for name in uploaded if name.lower().endswith('.csv')]
    if len(candidates) != 1:
        raise ValueError('Unggah tepat satu file CSV MOOC, lalu jalankan ulang sel ini.')
    file_name = candidates[0]
    raw_bytes = uploaded[file_name]
else:
    candidates = [Path(LOCAL_CSV)] if LOCAL_CSV else [p for p in [Path('MOOC(1).csv'), Path('MOOC.csv')] if p.exists()]
    if len(candidates) != 1:
        raise FileNotFoundError('Isi LOCAL_CSV dengan path satu file MOOC yang ingin dianalisis.')
    file_name = candidates[0].name
    raw_bytes = candidates[0].read_bytes()

for encoding in ['utf-8-sig', 'cp1252']:
    try:
        raw_bytes.decode(encoding, errors='strict')
        df_raw = pd.read_csv(BytesIO(raw_bytes), encoding=encoding, dtype=str, keep_default_na=False)
        break
    except UnicodeDecodeError:
        continue
else:
    raise ValueError('Encoding belum dikenali. Periksa file asli; jangan abaikan karakter rusak.')

expected_columns = ['Course Name', 'University / Industry Partner Name', 'Difficulty Level',
                    'Course Rating', 'Course URL', 'Course Description', 'all_skill']
if list(df_raw.columns) != expected_columns:
    raise ValueError(f'Struktur CSV berbeda dari MOOC yang diharapkan: {list(df_raw.columns)}')
assert len(df_raw) > 0, 'Dataset kosong.'
original_snapshot = df_raw.copy(deep=True)
sha256 = hashlib.sha256(raw_bytes).hexdigest()
print('File:', file_name, '| Encoding pembacaan:', encoding)
print(f'Ukuran: {len(raw_bytes):,} byte | Baris: {len(df_raw):,} | Kolom: {df_raw.shape[1]}')
print('SHA-256:', sha256)
display(df_raw.head())

## 3. Struktur dan kamus data
Seluruh kolom dibaca sebagai string agar nilai mentah tidak berubah. Rating numerik dihitung hanya pada variabel eksplorasi terpisah. Deskripsi atribut di bawah merupakan interpretasi berdasarkan nama kolom dan contoh isinya; bukan kamus resmi dari pembuat dataset.

In [ ]:
kamus = pd.DataFrame([
    ['Course Name', 'Nama kursus', 'Teks utama untuk representasi konten'],
    ['University / Industry Partner Name', 'Institusi atau mitra penyedia', 'Metadata tampilan dan eksplorasi penyedia'],
    ['Difficulty Level', 'Kategori tingkat kesulitan', 'Calon filter; kategori perlu diverifikasi'],
    ['Course Rating', 'Rating kursus; terdapat nilai nonnumerik', 'Metadata kualitas, bukan kemampuan siswa'],
    ['Course URL', 'Tautan kursus', 'Calon kunci identitas dan tautan referensi'],
    ['Course Description', 'Deskripsi kursus', 'Sumber teks utama korpus'],
    ['all_skill', 'Gabungan keterampilan dan label topik', 'Sumber teks pendukung; pemisah belum baku']
], columns=['kolom', 'makna', 'peran_dalam_tema'])
display(kamus)
df_raw.info()
display(pd.DataFrame({'dtype': df_raw.dtypes.astype(str),
                      'nilai_unik_mentah': df_raw.nunique(dropna=False)}))
display(df_raw.sample(min(5, len(df_raw)), random_state=42))

## 4. Audit kelengkapan — hanya mendeteksi
Sel kosong/whitespace diperiksa terpisah dari penanda seperti `NaN`, `null`, dan `N/A`. `Not Calibrated` tidak otomatis dianggap kosong: maknanya belum terkalibrasi dan harus dilaporkan tersendiri. Tidak ada nilai yang diisi atau dihapus.

In [ ]:
blank = df_raw.apply(lambda s: s.isna() | s.str.strip().eq(''))
markers = {'nan', 'null', 'none', 'n/a', 'na'}
marker_mask = df_raw.apply(lambda s: s.str.strip().str.lower().isin(markers))
unavailable = blank | marker_mask
quality = pd.DataFrame({
    'sel_kosong': blank.sum(),
    'penanda_missing_teks': marker_mask.sum(),
    'total_tidak_tersedia': unavailable.sum(),
    'persen_tidak_tersedia': (unavailable.mean()*100).round(2),
    'nilai_unik_nonmissing': [df_raw.loc[~unavailable[c], c].nunique() for c in df_raw.columns]
})
display(quality)
display(df_raw.loc[unavailable.any(axis=1)].head(8))
fig, ax = plt.subplots(figsize=(10, 5))
quality['total_tidak_tersedia'].sort_values().plot.barh(ax=ax, color='#dc8051')
ax.set(xlabel='Jumlah baris', title='Kelengkapan atribut pada data mentah')
fig.tight_layout(); fig.savefig(OUTPUT_DIR/'01_kelengkapan.png', dpi=160); plt.show()

## 5. Audit duplikasi dan identitas kursus
Duplikasi penuh dihitung sebagai jumlah baris berulang setelah kemunculan pertama. Nama sama belum tentu berarti kursus sama. URL berulang perlu dibandingkan dengan seluruh atribut sebelum menentukan deduplikasi pada langkah 2.

In [ ]:
full_duplicate = df_raw.duplicated(keep='first')
url_duplicate = (~unavailable['Course URL']) & df_raw['Course URL'].duplicated(keep='first')
name_duplicate = (~unavailable['Course Name']) & df_raw['Course Name'].duplicated(keep='first')
dup_summary = pd.DataFrame({'pemeriksaan': ['Duplikat penuh setelah baris pertama',
    'URL berulang setelah kemunculan pertama', 'Nama berulang setelah kemunculan pertama'],
    'jumlah': [int(full_duplicate.sum()), int(url_duplicate.sum()), int(name_duplicate.sum())]})
display(dup_summary)
display(df_raw.loc[df_raw.duplicated(keep=False)].head(10))
url_variants = df_raw.loc[~unavailable['Course URL']].groupby('Course URL')[
    [c for c in df_raw.columns if c != 'Course URL']].nunique(dropna=False)
conflicting_urls = url_variants.index[url_variants.gt(1).any(axis=1)]
print('URL dengan perbedaan atribut antarbaris:', len(conflicting_urls))
display(df_raw.loc[df_raw['Course URL'].isin(conflicting_urls)].head(10))

## 6. Distribusi tingkat kesulitan dan penyedia
Grafik memakai **seluruh baris mentah**, termasuk duplikasi; frekuensi belum mewakili kursus unik. `Conversant` dan `Not Calibrated` tidak dipetakan secara paksa ke Beginner/Intermediate/Advanced karena arti urutannya belum diketahui.

In [ ]:
def count_labels(column):
    # Label pengganti hanya untuk tampilan, tidak menulis ke df_raw.
    labels = df_raw[column].where(~unavailable[column], '(Tidak tersedia)')
    return labels.value_counts(dropna=False)

level_counts = count_labels('Difficulty Level')
provider_counts = count_labels('University / Industry Partner Name')
display(level_counts.rename_axis('level').reset_index(name='jumlah_baris'))
display(provider_counts.head(10).rename_axis('penyedia').reset_index(name='jumlah_baris'))
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
level_counts.sort_values().plot.barh(ax=axes[0], color='#287c8e')
provider_counts.head(10).sort_values().plot.barh(ax=axes[1], color='#5d66ac')
axes[0].set(title='Distribusi tingkat kesulitan', xlabel='Baris mentah')
axes[1].set(title='10 penyedia terbanyak', xlabel='Baris mentah')
fig.tight_layout(); fig.savefig(OUTPUT_DIR/'02_level_penyedia.png', dpi=160); plt.show()

## 7. Eksplorasi rating
Konversi numerik dilakukan pada **salinan analisis**. Nilai nonnumerik tetap utuh di data asli. Rentang 0–5 dipakai sebagai asumsi pemeriksaan rating berdasarkan tampilan nilai, bukan metadata resmi. Rating tidak boleh digunakan sebagai label kompetensi siswa.

In [ ]:
rating_numeric = pd.to_numeric(df_raw['Course Rating'], errors='coerce')
rating_nonnumeric = (~unavailable['Course Rating']) & rating_numeric.isna()
rating_outside = rating_numeric.notna() & ~rating_numeric.between(0, 5)
display(df_raw.loc[rating_nonnumeric, 'Course Rating'].value_counts().rename('jumlah'))
print('Rating kosong/penanda missing:', int(unavailable['Course Rating'].sum()))
print('Rating nonnumerik nonmissing:', int(rating_nonnumeric.sum()))
print('Rating di luar asumsi 0–5:', int(rating_outside.sum()))
display(rating_numeric.describe().to_frame('rating_numerik_untuk_eksplorasi'))
fig, ax = plt.subplots()
ax.hist(rating_numeric.dropna(), bins=20, color='#287c8e', edgecolor='white')
ax.set(title='Distribusi rating yang dapat dibaca sebagai angka', xlabel='Rating', ylabel='Baris mentah')
fig.tight_layout(); fig.savefig(OUTPUT_DIR/'03_rating.png', dpi=160); plt.show()

## 8. Eksplorasi teks korpus
Penghitungan karakter dan token whitespace merupakan ukuran deskriptif kasar, bukan tokenisasi model. Frekuensi kata di bawah sengaja mempertahankan huruf besar/kecil, tanda baca, dan stopword. Dengan demikian, hasilnya bukan daftar topik hasil NLP.

Kolom `all_skill` memuat frasa dengan spasi sekaligus label topik. Memisahkan berdasarkan setiap spasi dapat memecah keterampilan seperti “Machine Learning”; karena itu notebook belum mengubahnya menjadi daftar keterampilan.

In [ ]:
text_columns = ['Course Name', 'Course Description', 'all_skill']
text_stats = pd.DataFrame(index=df_raw.index)
for col in text_columns:
    text_stats[col + ' | karakter'] = df_raw[col].str.len()
    text_stats[col + ' | token_whitespace'] = df_raw[col].str.split().str.len()
display(text_stats.describe().T.round(2))
short_description = (~unavailable['Course Description']) & text_stats['Course Description | token_whitespace'].lt(20)
print('Deskripsi nonkosong <20 token (ambang eksplorasi, bukan aturan hapus):', int(short_description.sum()))
display(df_raw.loc[short_description, ['Course Name', 'Course Description']].head())
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col in zip(axes, ['Course Description', 'all_skill']):
    values = text_stats[col+' | token_whitespace']
    ax.hist(values, bins=40, color='#5d66ac', edgecolor='white')
    ax.set(title='Panjang '+col, xlabel='Token whitespace', ylabel='Baris mentah')
fig.tight_layout(); fig.savefig(OUTPUT_DIR/'04_panjang_teks.png', dpi=160); plt.show()
raw_tokens = Counter(token for text in df_raw['Course Description'] for token in text.split())
token_frequency = pd.DataFrame(raw_tokens.most_common(20), columns=['token_mentah', 'frekuensi'])
display(token_frequency)
for idx in df_raw.sample(min(3, len(df_raw)), random_state=42).index:
    print('\nBARIS CSV', idx+2)
    for col in text_columns:
        print(col + ':', df_raw.at[idx, col][:700])

## 9. Indikasi format teks dan validitas URL
Audit hanya mendeteksi pola HTML, newline, karakter pengganti, dan format URL. Ada tag HTML tidak otomatis berarti konten salah. Validitas format URL juga tidak membuktikan tautan masih aktif; notebook tidak mengakses situs eksternal. Dominan berbahasa Inggris berdasarkan sampel, sehingga kueri bahasa Indonesia kelak perlu strategi lintas bahasa yang dirancang terpisah.

In [ ]:
format_rows = []
for col in text_columns:
    s = df_raw[col]
    format_rows.append({
        'kolom': col,
        'indikasi_tag_html': int(s.str.contains(r'<[^>]+>', regex=True).sum()),
        'newline_aktual': int(s.str.contains('\n', regex=False).sum()),
        'literal_backslash_n': int(s.str.contains(r'\n', regex=False).sum()),
        'karakter_pengganti': int(s.str.contains('\ufffd', regex=False).sum()),
        'spasi_tepi': int(s.ne(s.str.strip()).sum())
    })
format_audit = pd.DataFrame(format_rows)
def inspect_url(value):
    try:
        p = urlparse(value)
        return p.scheme in {'http', 'https'} and bool(p.netloc), p.netloc.lower()
    except ValueError:
        return False, ''
url_info = df_raw['Course URL'].map(inspect_url)
url_valid = url_info.map(lambda x: x[0])
url_domains = url_info.map(lambda x: x[1]).value_counts()
display(format_audit)
print('URL dengan format tidak valid:', int((~url_valid).sum()))
display(url_domains.rename('jumlah_baris'))
display(df_raw.loc[~url_valid, ['Course Name', 'Course URL']].head())

## 10. Penyusunan korpus mentah domain
Korpus berupa tabel **satu dokumen per baris sumber** dengan nama, deskripsi, dan keterampilan tetap di kolom terpisah. `document_id` adalah ID lokal untuk pelacakan, bukan ID resmi kursus. Nomor baris CSV dimulai dari 2 karena baris pertama adalah header.

Tidak dilakukan penggabungan/normalisasi teks, pemilihan subset, deduplikasi, ataupun vektorisasi. Kolom `has_*` hanya mencatat ketersediaan teks; status lengkap bukan penilaian kualitas pedagogis.

In [ ]:
corpus_raw = df_raw.copy(deep=True)
corpus_raw.insert(0, 'source_csv_row', np.arange(2, len(df_raw)+2))
corpus_raw.insert(0, 'document_id', [f'MOOC-{i:05d}' for i in range(1, len(df_raw)+1)])
for col, short in [('Course Name', 'title'), ('Course Description', 'description'), ('all_skill', 'skills')]:
    corpus_raw['has_'+short] = ~unavailable[col]
complete_text = (~unavailable[text_columns]).all(axis=1)
any_text = (~unavailable[text_columns]).any(axis=1)
print('Dokumen dengan ketiga atribut teks tersedia:', int(complete_text.sum()))
print('Dokumen dengan minimal satu atribut teks tersedia:', int(any_text.sum()))
display(corpus_raw.head(5))
assert corpus_raw['document_id'].is_unique
assert len(corpus_raw) == len(df_raw)
pd.testing.assert_frame_equal(corpus_raw[expected_columns], original_snapshot)

## 11. Kesesuaian dataset, batasan, dan kebutuhan tahap selanjutnya
| Kebutuhan tema | Ketersediaan | Implikasi |
|---|---|---|
| Korpus nama/deskripsi/keterampilan | Ada, sebagian kosong | Dapat menjadi dasar rekomendasi berbasis konten setelah langkah 2 |
| Tingkat kesulitan | Ada lima kategori dan nilai kosong | Perlu verifikasi makna kategori sebelum membuat aturan level |
| Tautan kursus | Ada | Berguna sebagai referensi; belum diuji keaktifan maupun akses gratisnya |
| Isi modul, soal, kunci dan tujuan belajar rinci | Tidak ada kolom khusus | Katalog kursus bukan pengganti isi materi atau bank soal |
| Profil, nilai kuis, kelemahan dan riwayat siswa | Tidak tersedia | Belum cukup untuk personalisasi adaptif berbasis performa |
| Hubungan prasyarat/urutan pedagogis | Tidak tersedia | Jalur belajar tidak boleh dianggap valid hanya dari kemiripan teks |
| Progres, XP, quest dan badge | Tidak tersedia | Dirancang pada langkah 4 menggunakan data interaksi aplikasi |

**Kesimpulan kelayakan:** dataset layak sebagai **korpus awal rekomendasi kursus berbasis konten**, tetapi belum merupakan dataset lengkap untuk rekomendasi remedial dan jalur belajar adaptif. Kelompok perlu menentukan apakah unit rekomendasi adalah kursus atau modul, serta menyiapkan tujuan belajar, prasyarat yang divalidasi, dan profil/hasil belajar siswa pada pengembangan berikutnya. Data simulasi, jika nanti digunakan, harus diberi label simulasi.

Untuk langkah 2 kelak, rencanakan penanganan deskripsi/keterampilan kosong, duplikasi berdasarkan identitas yang tepat, encoding/format teks, kategori belum terkalibrasi, serta strategi bahasa. **Belum ada keputusan pembersihan yang diterapkan di notebook ini.**

In [ ]:
summary = {
    'nama_file': file_name, 'sha256': sha256, 'encoding_pembacaan': encoding,
    'baris_mentah': len(df_raw), 'kolom_asli': len(expected_columns),
    'url_unik_nonmissing': int(df_raw.loc[~unavailable['Course URL'], 'Course URL'].nunique()),
    'duplikat_penuh_setelah_pertama': int(full_duplicate.sum()),
    'url_dengan_konflik_atribut': len(conflicting_urls),
    'missing_per_kolom': {c: int(unavailable[c].sum()) for c in expected_columns},
    'rating_nonnumerik_nonmissing': int(rating_nonnumeric.sum()),
    'dokumen_teks_lengkap': int(complete_text.sum()),
    'dokumen_minimal_satu_teks': int(any_text.sum()),
    'status': 'Langkah 1 saja; data mentah belum dibersihkan atau divektorisasi',
    'provenans': 'File dari kelompok; URL sumber unduhan, pembuat, tanggal pengumpulan dan lisensi belum diverifikasi',
    'versi': {'python': platform.python_version(), 'pandas': pd.__version__, 'numpy': np.__version__}
}
report = f"""# Ringkasan Langkah 1 — Eksplorasi Korpus MOOC

Dataset {file_name} berisi {len(df_raw):,} baris dan {len(expected_columns)} atribut.
Unit observasi adalah baris katalog kursus, bukan siswa. Terdapat {summary['url_unik_nonmissing']:,} URL unik
serta {int(full_duplicate.sum())} baris duplikat penuh setelah kemunculan pertama.
Seluruh frekuensi eksplorasi dihitung dari data mentah, sehingga duplikasi masih memengaruhi distribusi.

Ketidaktersediaan data per atribut: {summary['missing_per_kolom']}.
Terdapat {int(rating_nonnumeric.sum())} rating nonnumerik nonmissing.
Sebanyak {int(complete_text.sum())} dokumen memiliki nama, deskripsi, dan keterampilan yang tersedia.

Dataset dapat digunakan sebagai korpus awal rekomendasi berbasis konten.
Belum tersedia profil siswa, hasil kuis, isi modul rinci, maupun relasi prasyarat.
Karena itu, personalisasi adaptif dan jalur belajar pedagogis belum dapat ditentukan hanya dari CSV ini.
Kategori Conversant dan Not Calibrated memerlukan validasi, bukan pemetaan level otomatis.

Korpus mentah mempertahankan seluruh baris dan atribut asli, dengan ID lokal dan indikator ketersediaan.
Tidak dilakukan pembersihan, imputasi, normalisasi teks, TF-IDF, cosine similarity, atau pemodelan.
Sumber dan lisensi CSV perlu dilengkapi oleh kelompok. Hash sumber: {sha256}.
"""
display(Markdown(report))

## 12. Ekspor hasil langkah 1
ZIP berisi korpus mentah, kamus data, tabel audit, statistik teks, frekuensi kategori/token mentah, ringkasan JSON/Markdown, dan empat grafik. **Bukan dataset bersih dan bukan model.** Korpus diekspor sebagai UTF-8 dengan BOM agar mudah dibaca kembali; perubahan encoding keluaran tidak mengubah isi teks. File mentah yang diunggah tidak ditimpa.

In [ ]:
pd.testing.assert_frame_equal(df_raw, original_snapshot)
corpus_raw.to_csv(OUTPUT_DIR/'korpus_mooc_mentah.csv', index=False, encoding='utf-8-sig')
kamus.to_csv(OUTPUT_DIR/'kamus_data.csv', index=False, encoding='utf-8-sig')
quality.to_csv(OUTPUT_DIR/'audit_kelengkapan.csv', encoding='utf-8-sig')
dup_summary.to_csv(OUTPUT_DIR/'audit_duplikasi.csv', index=False, encoding='utf-8-sig')
format_audit.to_csv(OUTPUT_DIR/'audit_format.csv', index=False, encoding='utf-8-sig')
text_stats.describe().T.to_csv(OUTPUT_DIR/'statistik_panjang_teks.csv', encoding='utf-8-sig')
level_counts.to_csv(OUTPUT_DIR/'distribusi_level.csv', encoding='utf-8-sig')
provider_counts.to_csv(OUTPUT_DIR/'distribusi_penyedia.csv', encoding='utf-8-sig')
token_frequency.to_csv(OUTPUT_DIR/'frekuensi_token_mentah.csv', index=False, encoding='utf-8-sig')
(OUTPUT_DIR/'ringkasan.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
(OUTPUT_DIR/'ringkasan_langkah_1.md').write_text(report, encoding='utf-8')
# Verifikasi ekspor tidak mengubah nilai asli.
reloaded = pd.read_csv(OUTPUT_DIR/'korpus_mooc_mentah.csv', dtype=str, keep_default_na=False, encoding='utf-8-sig')
pd.testing.assert_frame_equal(reloaded[expected_columns], original_snapshot)
zip_path = shutil.make_archive('Hasil_Langkah_1_MOOC', 'zip', OUTPUT_DIR)
print('SELESAI: data asli tetap utuh. Hasil tersedia di:', zip_path)
print('File hasil:', *[p.name for p in sorted(OUTPUT_DIR.iterdir())], sep='\n- ')

In [ ]:
print("Hasil langkah 1 tersedia. Lanjutkan langkah 2–4; ZIP lengkap diunduh pada akhir notebook.")

**Langkah 1 selesai.** Data mentah tetap utuh. Bagian berikutnya bekerja pada salinan `df_clean`.

# Langkah 2 — Preprocessing & Transformasi
## 1. Pembersihan data
Pertahankan kolom sumber. Teks kosong diisi string kosong untuk penggabungan korpus; materi tanpa teks informatif tidak digunakan. Duplikat identik dihapus. Rating nonnumerik menjadi nilai kosong pada kolom numerik tambahan, tidak diisi dengan nilai buatan. Level Conversant dan Not Calibrated tidak dipaksakan menjadi level lain. URL identik dengan isi berbeda perlu diperiksa manual.

In [ ]:
df_clean = df_raw.copy(deep=True)
text_columns = ['Course Name', 'Course Description', 'all_skill']
for col in df_clean.columns:
    missing = df_clean[col].str.strip().str.lower().isin(['', 'nan', 'null', 'none', 'n/a', 'na'])
    df_clean.loc[missing, col] = ''
jumlah_duplikat = int(df_clean.duplicated().sum())
df_clean = df_clean.drop_duplicates().reset_index(drop=True)
if df_clean.loc[df_clean['Course URL'].ne(''), 'Course URL'].duplicated().any():
    raise ValueError('Ada URL sama dengan atribut berbeda. Periksa sebelum melanjutkan.')
df_clean['course_id'] = ['MOOC-' + hashlib.sha256(url.encode()).hexdigest()[:16]
                         if url else f'ROW-{i:05d}' for i, url in enumerate(df_clean['Course URL'])]
assert df_clean['course_id'].is_unique
df_clean['rating_numeric'] = pd.to_numeric(df_clean['Course Rating'], errors='coerce')
df_clean.loc[~df_clean['rating_numeric'].between(0, 5), 'rating_numeric'] = np.nan
print('Duplikat dihapus:', jumlah_duplikat)
print('Jumlah kursus:', len(df_clean))
display(df_clean[text_columns].eq('').sum().to_frame('teks_kosong'))

## 2. Normalisasi teks
Gunakan normalisasi yang sama untuk data dan kueri: huruf kecil, HTML entity/tag dan newline dibersihkan, tanda pemisah menjadi spasi. Angka dan huruf Unicode dipertahankan. `C++` → `cplusplus`, `C#` → `csharp`, `.NET` → `dotnet` agar istilah teknis tetap berbeda. Stopword Inggris diabaikan pada vectorizer, karena sampel korpus didominasi bahasa Inggris. Kueri contoh menggunakan bahasa Inggris; penerjemahan otomatis tidak disediakan.

In [ ]:
import html
import re
import unicodedata

def normalisasi_teks(teks):
    if not isinstance(teks, str):
        raise TypeError('Teks harus berupa string.')
    teks = unicodedata.normalize('NFKC', html.unescape(teks)).lower()
    teks = teks.replace('\\n', ' ').replace('\\r', ' ').replace('\\t', ' ')
    teks = re.sub(r'<[^>]+>', ' ', teks)
    teks = re.sub(r'(?<!\w)c\+\+(?!\w)', ' cplusplus ', teks)
    teks = re.sub(r'(?<!\w)c#(?!\w)', ' csharp ', teks)
    teks = re.sub(r'(?<!\w)\.net\b', ' dotnet ', teks)
    teks = re.sub(r'[^\w\s]', ' ', teks, flags=re.UNICODE).replace('_', ' ')
    return re.sub(r'\s+', ' ', teks).strip()

In [ ]:
for col in text_columns:
    df_clean[col + '_normalized'] = df_clean[col].apply(normalisasi_teks)
df_clean['text_corpus'] = df_clean[[c+'_normalized' for c in text_columns]].agg(' '.join, axis=1).str.strip()
jumlah_tanpa_teks = int(df_clean['text_corpus'].eq('').sum())
df_clean = df_clean.loc[df_clean['text_corpus'].ne('')].reset_index(drop=True)
print('Baris tanpa teks dikeluarkan:', jumlah_tanpa_teks)
display(df_clean[['Course Name', 'text_corpus']].head())

## 3. TF-IDF
Vectorizer dilatih satu kali pada korpus; kueri hanya menggunakan `transform`. Matriks tetap sparse. Tidak menggunakan StandardScaler karena alur rekomendasi menggunakan bobot TF-IDF dengan normalisasi L2. Tampilan hanya mengambil bagian kecil matriks agar hemat RAM.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf_vectorizer = TfidfVectorizer(stop_words='english', token_pattern=r'(?u)\b\w+\b', norm='l2')
tfidf_matrix = tfidf_vectorizer.fit_transform(df_clean['text_corpus'])
# Buang dokumen yang seluruh tokennya diabaikan vectorizer, jika ada.
nonzero_rows = np.asarray(tfidf_matrix.getnnz(axis=1)).ravel() > 0
jumlah_vektor_nol = int((~nonzero_rows).sum())
if jumlah_vektor_nol:
    df_clean = df_clean.loc[nonzero_rows].reset_index(drop=True)
    tfidf_matrix = tfidf_vectorizer.fit_transform(df_clean['text_corpus'])
feature_names = tfidf_vectorizer.get_feature_names_out()
print('Ukuran matriks TF-IDF:', tfidf_matrix.shape)
print('Dokumen tanpa fitur dikeluarkan:', jumlah_vektor_nol)
display(pd.DataFrame(tfidf_matrix[:5, :20].toarray(), columns=feature_names[:20]))
assert len(df_clean) == tfidf_matrix.shape[0]

# Langkah 3 — Pemodelan & Implementasi Algoritma Cerdas
## 1. Rekomendasi kursus dan pencarian
Cosine similarity mengukur kemiripan teks, bukan nilai kuis atau probabilitas keberhasilan. Skor nol tidak dimasukkan ke rekomendasi. Jumlah hasil dan indeks divalidasi; kursus acuan tidak direkomendasikan kembali. Fuzzy Matching dan Rabin-Karp dari versi sebelumnya dihilangkan agar implementasi fokus pada algoritma tema: **TF-IDF + cosine similarity**.

In [ ]:
def validasi_jumlah(jumlah):
    if isinstance(jumlah, (bool, np.bool_)) or not isinstance(jumlah, (int, np.integer)) or jumlah < 1:
        raise ValueError('Jumlah hasil harus bilangan bulat positif.')

def hitung_cosine_similarity(indeks_kursus):
    if isinstance(indeks_kursus, (bool, np.bool_)) or not isinstance(indeks_kursus, (int, np.integer)):
        raise ValueError('Indeks kursus harus bilangan bulat.')
    if not 0 <= indeks_kursus < len(df_clean):
        raise IndexError('Indeks kursus tidak ditemukan.')
    return cosine_similarity(tfidf_matrix[indeks_kursus], tfidf_matrix).ravel()

kolom_tampil = ['course_id', 'Course Name', 'Difficulty Level', 'Course Rating', 'Course URL']
def urutkan_hasil(skor, jumlah, mask=None):
    validasi_jumlah(jumlah)
    eligible = np.asarray(skor) > 0
    if mask is not None:
        eligible &= np.asarray(mask, dtype=bool)
    indices = np.flatnonzero(eligible)
    indices = indices[np.argsort(-skor[indices], kind='stable')][:jumlah]
    hasil = df_clean.iloc[indices][kolom_tampil].copy()
    hasil['Skor Cosine Similarity'] = skor[indices]
    return hasil.reset_index(drop=True)

def rekomendasi_kursus(indeks_kursus, jumlah_rekomendasi=5):
    skor = hitung_cosine_similarity(indeks_kursus)
    skor[indeks_kursus] = -1
    return urutkan_hasil(skor, jumlah_rekomendasi)

def pencarian_kursus(kata_kunci, jumlah_hasil=5):
    validasi_jumlah(jumlah_hasil)
    query = normalisasi_teks(kata_kunci)
    if not query:
        raise ValueError('Kata kunci tidak boleh kosong.')
    vector = tfidf_vectorizer.transform([query])
    skor = cosine_similarity(vector, tfidf_matrix).ravel()
    return urutkan_hasil(skor, jumlah_hasil)

hasil_rekomendasi = rekomendasi_kursus(0)
hasil_pencarian = pencarian_kursus('python programming and data analysis')
display(hasil_rekomendasi)
display(hasil_pencarian)

## 2. Aturan adaptasi rekomendasi
Aturan berikut adalah **asumsi prototipe kelompok**, bukan hasil pelatihan atau ketentuan ambang nilai dari dosen. Nilai kuis harus terkait topik yang dicari dan level kemampuan diinput oleh siswa/pengajar. `Conversant`, `Not Calibrated`, dan level kosong tetap tersimpan di katalog tetapi tidak dipilih untuk jalur yang memerlukan level pasti.

| Nilai kuis topik | Quest | Level materi sasaran |
|---|---|---|
| <60 | Remedial | Turun satu level, minimal Beginner |
| 60–<80 | Improvement | Tetap di level saat ini |
| ≥80 | Mastery | Naik satu level, maksimal Advanced |

Sistem menyaring level lalu mengurutkan kemiripan dengan topik. Kursus selesai dikecualikan. Jika tidak ada kandidat relevan pada level sasaran, hasil kosong dilaporkan; sistem tidak memaksakan materi level lain. Urutan ini adalah urutan rekomendasi prototipe, **bukan bukti relasi prasyarat**. Validasi materi/prasyarat oleh pengajar masih diperlukan.

In [ ]:
from numbers import Real
LEVEL_MATERI = ['Beginner', 'Intermediate', 'Advanced']

def validasi_nilai(nilai_kuis):
    if isinstance(nilai_kuis, (bool, np.bool_)) or not isinstance(nilai_kuis, Real):
        raise ValueError('Nilai kuis harus angka 0–100.')
    if not np.isfinite(nilai_kuis) or not 0 <= nilai_kuis <= 100:
        raise ValueError('Nilai kuis harus angka 0–100.')

def tentukan_quest(nilai_kuis):
    validasi_nilai(nilai_kuis)
    if nilai_kuis < 60:
        return 'Remedial'
    if nilai_kuis < 80:
        return 'Improvement'
    return 'Mastery'

def level_sasaran(nilai_kuis, level_saat_ini):
    validasi_nilai(nilai_kuis)
    if level_saat_ini not in LEVEL_MATERI:
        raise ValueError('Level kemampuan harus Beginner, Intermediate, atau Advanced.')
    idx = LEVEL_MATERI.index(level_saat_ini)
    delta = -1 if nilai_kuis < 60 else (1 if nilai_kuis >= 80 else 0)
    return LEVEL_MATERI[max(0, min(2, idx+delta))]

def rekomendasi_adaptif(topik, nilai_kuis, level_saat_ini, kursus_selesai=None, jumlah_hasil=5):
    target = level_sasaran(nilai_kuis, level_saat_ini)
    validasi_jumlah(jumlah_hasil)
    query = normalisasi_teks(topik)
    if not query:
        raise ValueError('Topik belajar tidak boleh kosong.')
    skor = cosine_similarity(tfidf_vectorizer.transform([query]), tfidf_matrix).ravel()
    mask = df_clean['Difficulty Level'].eq(target) & ~df_clean['course_id'].isin(kursus_selesai or [])
    hasil = urutkan_hasil(skor, jumlah_hasil, mask)
    hasil['Quest'] = tentukan_quest(nilai_kuis)
    hasil['Urutan'] = np.arange(1, len(hasil)+1)
    return hasil

for nilai in [45, 70, 90]:
    print('Nilai simulasi:', nilai, '| Level sasaran:', level_sasaran(nilai, 'Intermediate'))
    display(rekomendasi_adaptif('python programming', nilai, 'Intermediate', jumlah_hasil=3))

# Langkah 4 — Integrasi Mekanik Gamifikasi
## 1. XP, level aktivitas, badge dan progres
Rumus dasar kelompok dipertahankan: materi selesai +20 XP; nilai ≥90 +50, ≥80 +30, ≥60 +20, selain itu +10. Badge sesuai nilai dan penyelesaian materi. Level berdasarkan total XP adalah **level aktivitas**, bukan bukti kemampuan atau pengganti level materi.

Reward hanya diberikan pada penyelesaian pertama course_id dalam simulasi siswa. Pengulangan tidak menambah XP. Progres dihitung terhadap daftar quest aktif yang tetap; rekomendasi berikutnya ditampilkan terpisah agar penyebut progres tidak berubah saat satu materi selesai.

In [ ]:
def hitung_xp(nilai_kuis, materi_selesai):
    validasi_nilai(nilai_kuis)
    if not isinstance(materi_selesai, bool):
        raise ValueError('materi_selesai harus True/False.')
    xp = 20 if materi_selesai else 0
    xp += 50 if nilai_kuis >= 90 else 30 if nilai_kuis >= 80 else 20 if nilai_kuis >= 60 else 10
    return xp

def tentukan_level(total_xp):
    if isinstance(total_xp, bool) or not isinstance(total_xp, (int, np.integer)) or total_xp < 0:
        raise ValueError('Total XP harus bilangan bulat nonnegatif.')
    if total_xp >= 300: return 'Level Aktivitas 4'
    if total_xp >= 200: return 'Level Aktivitas 3'
    if total_xp >= 100: return 'Level Aktivitas 2'
    return 'Level Aktivitas 1'

def tentukan_badge(nilai_kuis, materi_selesai):
    validasi_nilai(nilai_kuis)
    if not isinstance(materi_selesai, bool):
        raise ValueError('materi_selesai harus True/False.')
    badges = []
    if nilai_kuis >= 90: badges.append('Quiz Master')
    if nilai_kuis >= 80: badges.append('Smart Learner')
    if materi_selesai: badges.append('Learning Explorer')
    if nilai_kuis < 60: badges.append('Keep Learning')
    return badges

def hitung_progress(materi_selesai, total_materi):
    for value in [materi_selesai, total_materi]:
        if isinstance(value, bool) or not isinstance(value, (int, np.integer)) or value < 0:
            raise ValueError('Jumlah materi harus bilangan bulat nonnegatif.')
    if materi_selesai > total_materi:
        raise ValueError('Jumlah selesai tidak boleh melebihi total.')
    return round(materi_selesai/total_materi*100, 2) if total_materi else 0.0

In [ ]:
def buat_profil(nama):
    if not isinstance(nama, str) or not nama.strip():
        raise ValueError('Nama siswa wajib diisi.')
    return {'nama': nama.strip(), 'total_xp': 0, 'kursus_selesai': [], 'badges': [], 'riwayat': []}

def mulai_jalur(profil, topik, nilai_awal, level_awal, jumlah=3):
    daftar = rekomendasi_adaptif(topik, nilai_awal, level_awal, profil['kursus_selesai'], jumlah)
    return {'topik': topik, 'level_materi': level_sasaran(nilai_awal, level_awal),
            'quest': tentukan_quest(nilai_awal), 'course_ids': daftar['course_id'].tolist(),
            'daftar': daftar}

def status_jalur(profil, jalur):
    tabel = jalur['daftar'].copy()
    tabel['Selesai'] = tabel['course_id'].isin(profil['kursus_selesai'])
    progress = hitung_progress(int(tabel['Selesai'].sum()), len(tabel))
    return tabel, progress

def selesaikan_materi(profil, jalur, course_id, nilai_kuis):
    validasi_nilai(nilai_kuis)
    if course_id not in jalur['course_ids']:
        raise ValueError('Kursus bukan bagian dari quest aktif.')
    if course_id in profil['kursus_selesai']:
        return {'status': 'Sudah tercatat; XP tidak ditambah', 'xp_diperoleh': 0}
    xp = hitung_xp(nilai_kuis, True)
    profil['total_xp'] += xp
    profil['kursus_selesai'].append(course_id)
    profil['badges'] = sorted(set(profil['badges'] + tentukan_badge(nilai_kuis, True)))
    profil['riwayat'].append({'course_id': course_id, 'topik': jalur['topik'],
        'nilai_kuis': float(nilai_kuis), 'xp': xp, 'sumber_nilai': 'simulasi'})
    berikutnya = rekomendasi_adaptif(jalur['topik'], nilai_kuis, jalur['level_materi'],
                                   profil['kursus_selesai'], jumlah_hasil=3)
    return {'status': 'Penyelesaian tercatat', 'xp_diperoleh': xp,
            'quest_berikutnya': tentukan_quest(nilai_kuis), 'rekomendasi_berikutnya': berikutnya}

# SIMULASI: nilai ini dibuat untuk demonstrasi, bukan berasal dari MOOC.
profil = buat_profil('Alya')
jalur = mulai_jalur(profil, 'python programming', nilai_awal=45, level_awal='Intermediate')
display(jalur['daftar'])
if jalur['course_ids']:
    hasil_gamifikasi = selesaikan_materi(profil, jalur, jalur['course_ids'][0], nilai_kuis=85)
    print('XP diperoleh:', hasil_gamifikasi['xp_diperoleh'])
    print('Total XP:', profil['total_xp'], '|', tentukan_level(profil['total_xp']))
    print('Badges:', profil['badges'])
    print('Quest berikutnya:', hasil_gamifikasi['quest_berikutnya'])
    display(hasil_gamifikasi['rekomendasi_berikutnya'])
else:
    print('Tidak ada kandidat relevan pada level sasaran. Tinjau topik/materi bersama pengajar.')
tabel_progress, progress = status_jalur(profil, jalur)
display(tabel_progress)
fig, ax = plt.subplots(figsize=(8, 2))
ax.barh(['Quest aktif'], [progress], color='#287c8e')
ax.set_xlim(0, 100); ax.set_xlabel('Progres (%)'); ax.set_title(f'Progres: {progress}%')
fig.tight_layout(); plt.show()

## 2. Pengujian logika langkah 1–4
Pengujian ini mencakup batas nilai, input tidak valid, rekomendasi tanpa hasil, pengecualian kursus selesai, dan hadiah berulang. Ini **bukan pengujian web end-to-end**; minimal 12 test case web pada panduan tetap dikerjakan setelah langkah 5–6.

Tabel evaluasi relevansi di bawah disediakan untuk diisi pengajar/kelompok secara manual. Belum ada label relevansi resmi; notebook tidak mengklaim akurasi atau Precision@K yang belum diukur.

In [ ]:
hasil_uji = []
def cek(nama, kondisi):
    assert bool(kondisi), nama
    hasil_uji.append({'pengujian': nama, 'hasil': 'LULUS'})
def cek_error(nama, fungsi):
    try:
        fungsi()
    except (ValueError, TypeError, IndexError):
        hasil_uji.append({'pengujian': nama, 'hasil': 'LULUS'})
    else:
        raise AssertionError(nama)

cek('Normalisasi istilah teknis', normalisasi_teks('C++ C# Python 3 HTML5') == 'cplusplus csharp python 3 html5')
cek('Pemisah kata', normalisasi_teks('data-driven') == 'data driven')
cek('Data asli tidak berubah', df_raw.equals(original_snapshot))
cek('Urutan data dan matriks cocok', len(df_clean) == tfidf_matrix.shape[0])
cek_error('Nilai negatif ditolak', lambda: hitung_xp(-5, True))
cek_error('Nilai di atas 100 ditolak', lambda: hitung_xp(101, True))
cek_error('Nilai NaN ditolak', lambda: hitung_xp(float('nan'), True))
cek_error('Progres melebihi total ditolak', lambda: hitung_progress(6, 5))
cek('Progres kosong', hitung_progress(0, 0) == 0)
cek('Batas nilai 60', tentukan_quest(60) == 'Improvement')
cek('Batas nilai 80', tentukan_quest(80) == 'Mastery')
cek('XP nilai 90 dan materi selesai', hitung_xp(90, True) == 70)
cek_error('Kueri kosong ditolak', lambda: pencarian_kursus('   '))
cek_error('Jumlah hasil nol ditolak', lambda: pencarian_kursus('python', 0))
cek_error('Indeks tidak valid ditolak', lambda: rekomendasi_kursus(-1))
cek('Kosakata tidak dikenal menghasilkan kosong', pencarian_kursus('zzzzqvxxx987654321').empty)
cek('Kursus acuan dikecualikan', df_clean.iloc[0]['course_id'] not in rekomendasi_kursus(0)['course_id'].tolist())
cek('Target remedial Beginner', level_sasaran(40, 'Intermediate') == 'Beginner')
cek('Target mastery Advanced', level_sasaran(90, 'Intermediate') == 'Advanced')
test_profil = buat_profil('Uji')
test_jalur = mulai_jalur(test_profil, 'python programming', 45, 'Intermediate')
cek('Ada kandidat uji remedial', len(test_jalur['course_ids']) > 0)
cid = test_jalur['course_ids'][0]
selesaikan_materi(test_profil, test_jalur, cid, 85)
previous_xp = test_profil['total_xp']
cek('Reward tidak berulang', selesaikan_materi(test_profil, test_jalur, cid, 85)['xp_diperoleh'] == 0 and test_profil['total_xp'] == previous_xp)
cek('Kursus selesai dikecualikan', cid not in rekomendasi_adaptif('python programming', 45, 'Intermediate', [cid])['course_id'].tolist())
cek_error('Kursus di luar quest ditolak', lambda: selesaikan_materi(test_profil, test_jalur, 'INVALID', 90))
uji_df = pd.DataFrame(hasil_uji)
display(uji_df)

evaluasi = []
for query in ['python programming', 'machine learning', 'sql database']:
    result = pencarian_kursus(query, 5)
    result.insert(0, 'query', query)
    result['relevan_0_atau_1'] = ''
    result['catatan_penilai'] = ''
    evaluasi.append(result)
evaluasi_relevansi = pd.concat(evaluasi, ignore_index=True)
display(evaluasi_relevansi)
print('Isi label relevansi manual sebelum menghitung Precision@K. Jumlah hasil bukan ukuran akurasi.')

## 3. Penyimpanan model dan hasil
Simpan vectorizer, matriks sparse, dan katalog dalam satu `.joblib` agar indeks kursus selaras. Fungsi normalisasi disertakan sebagai file Python terpisah untuk dipakai sebelum `transform` pada aplikasi nanti. Catatan versi disimpan untuk reproduksibilitas. Hanya muat file joblib dari sumber tepercaya.

State siswa disimpan sebagai JSON untuk demonstrasi; penyimpanan per pengguna dan validasi di server baru diintegrasikan pada langkah 6. Tidak ada klaim bahwa Colab sudah merupakan aplikasi web.

In [ ]:
import joblib
import sklearn
OUTPUT_MODEL = Path('hasil_langkah_2_4')
OUTPUT_MODEL.mkdir(exist_ok=True)
model_bundle = {'vectorizer': tfidf_vectorizer, 'matrix': tfidf_matrix, 'courses': df_clean,
                'sklearn_version': sklearn.__version__, 'source_sha256': sha256,
                'level_materi': LEVEL_MATERI}
model_path = OUTPUT_MODEL/'model_rekomendasi_mooc.joblib'
joblib.dump(model_bundle, model_path, compress=3)
NORMALIZER_SOURCE = "import html\nimport re\nimport unicodedata\n\ndef normalisasi_teks(teks):\n    if not isinstance(teks, str):\n        raise TypeError('Teks harus berupa string.')\n    teks = unicodedata.normalize('NFKC', html.unescape(teks)).lower()\n    teks = teks.replace('\\\\n', ' ').replace('\\\\r', ' ').replace('\\\\t', ' ')\n    teks = re.sub(r'<[^>]+>', ' ', teks)\n    teks = re.sub(r'(?<!\\w)c\\+\\+(?!\\w)', ' cplusplus ', teks)\n    teks = re.sub(r'(?<!\\w)c#(?!\\w)', ' csharp ', teks)\n    teks = re.sub(r'(?<!\\w)\\.net\\b', ' dotnet ', teks)\n    teks = re.sub(r'[^\\w\\s]', ' ', teks, flags=re.UNICODE).replace('_', ' ')\n    return re.sub(r'\\s+', ' ', teks).strip()\n"
(OUTPUT_MODEL/'normalisasi_teks.py').write_text(NORMALIZER_SOURCE, encoding='utf-8')
# Uji hasil setelah dimuat kembali.
loaded = joblib.load(model_path)
query_check = normalisasi_teks('python programming')
before = cosine_similarity(tfidf_vectorizer.transform([query_check]), tfidf_matrix).ravel()
after = cosine_similarity(loaded['vectorizer'].transform([query_check]), loaded['matrix']).ravel()
assert np.allclose(before, after)
assert loaded['courses']['course_id'].tolist() == df_clean['course_id'].tolist()
print('Uji simpan/muat model: LULUS')
df_clean.to_csv(OUTPUT_MODEL/'mooc_clean.csv', index=False, encoding='utf-8-sig')
hasil_rekomendasi.to_csv(OUTPUT_MODEL/'contoh_rekomendasi.csv', index=False, encoding='utf-8-sig')
tabel_progress.to_csv(OUTPUT_MODEL/'quest_aktif_simulasi.csv', index=False, encoding='utf-8-sig')
uji_df.to_csv(OUTPUT_MODEL/'hasil_pengujian_logika.csv', index=False, encoding='utf-8-sig')
evaluasi_relevansi.to_csv(OUTPUT_MODEL/'evaluasi_relevansi_belum_dinilai.csv', index=False, encoding='utf-8-sig')
(OUTPUT_MODEL/'profil_simulasi.json').write_text(json.dumps(profil, ensure_ascii=False, indent=2), encoding='utf-8')
metadata = {'python': platform.python_version(), 'pandas': pd.__version__, 'sklearn': sklearn.__version__,
            'jumlah_kursus': len(df_clean), 'fitur_tfidf': tfidf_matrix.shape[1],
            'sumber': file_name, 'sha256': sha256, 'nilai_kuis': 'simulasi',
            'status_validasi_pedagogis': 'belum divalidasi pengajar',
            'aturan_adaptasi': 'nilai <60 turun satu level; 60–<80 tetap; >=80 naik satu level'}
(OUTPUT_MODEL/'metadata.json').write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')
(OUTPUT_MODEL/'README.txt').write_text(
    'Prototipe ITS edukatif langkah 1–4. Bukan penilaian resmi kemampuan siswa.\n'
    'Gunakan normalisasi_teks.py untuk kueri, lalu vectorizer.transform dan cosine similarity.\n'
    'Jaga keselarasan urutan courses dan matrix dalam model_rekomendasi_mooc.joblib.\n'
    'Nilai/profil siswa simulasi. Relasi prasyarat dan relevansi memerlukan validasi pengajar.\n'
    'Langkah 5–6: HTML/CSS/JS, REST API, validasi input server, academic disclaimer, dan pengujian web.\n', encoding='utf-8')
zip_name = 'Hasil_ITS_Langkah_1_4.zip'
from zipfile import ZipFile, ZIP_DEFLATED
with ZipFile(zip_name, 'w', ZIP_DEFLATED) as archive:
    for folder in [Path('hasil_langkah_1'), OUTPUT_MODEL]:
        for path in sorted(folder.iterdir()):
            if path.is_file(): archive.write(path, arcname=str(path))
print('Semua hasil disimpan:', zip_name)

In [ ]:
if IN_COLAB:
    files.download(zip_name)
else:
    print('Unduh ZIP dari folder kerja:', Path(zip_name).resolve())

## Kesimpulan
Langkah 1 mengeksplorasi katalog MOOC. Langkah 2 membersihkan data dan menghasilkan TF-IDF. Langkah 3 menggunakan cosine similarity serta aturan nilai/level untuk memilih kandidat materi. Langkah 4 mengaitkan materi rekomendasi dengan quest aktif, pencatatan penyelesaian, XP, badge, dan progres.

Pengujian logika serta simpan/muat model tersedia. Relevansi rekomendasi, kategori kesulitan sumber dan prasyarat pedagogis masih memerlukan penilaian pengajar. Data nilai siswa adalah simulasi. Untuk menyelesaikan seluruh tubes, lanjutkan langkah 5–6 sesuai panduan; notebook ini berhenti pada langkah 4.

**Acuan:** Panduan dan Ketentuan Tubes ITS yang dilampirkan, bagian B, C, dan D. Lengkapi nama anggota, sumber unduhan CSV, lisensi, serta hasil evaluasi relevansi sebelum pengumpulan.